# Rede neural manual — classificação de duas luas

Projeto da disciplina Matemática para Ciência de Dados, baseado no `exemplo4.py` da aula de 21/08/2026. A rede abaixo é implementada com NumPy, sem Keras, TensorFlow, PyTorch ou modelos prontos do scikit-learn.

**Modificações:** arquitetura 2→8→4→1, duas camadas ocultas com `tanh`, entropia cruzada binária, inicialização de Xavier e avaliação em dados de teste.

In [ ]:
# No Colab, estas bibliotecas normalmente já estão instaladas.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

## 1. Dados

O scikit-learn é usado somente para gerar e separar os dados. A média e o desvio são calculados exclusivamente no treino para evitar vazamento de informação.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.22, random_state=SEED)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
media, desvio = X_treino.mean(axis=0), X_treino.std(axis=0)
X_treino = (X_treino - media) / desvio
X_teste = (X_teste - media) / desvio

plt.figure(figsize=(6, 4))
plt.scatter(X_treino[:, 0], X_treino[:, 1], c=y_treino, cmap='bwr', edgecolors='white')
plt.title('Dados de treino: duas luas')
plt.xlabel('Atributo 1'); plt.ylabel('Atributo 2'); plt.show()

## 2. Rede neural

A propagação direta calcula `Z = A @ W + b`. Nas camadas ocultas, `A = tanh(Z)`; na saída, `A = sigmoid(Z)`. Na retropropagação, a combinação de sigmoide e entropia cruzada produz o gradiente inicial `delta = y_pred - y`.

In [ ]:
class RedeNeuralManual:
    def __init__(self, arquitetura=(2, 8, 4, 1), taxa=0.05, seed=42):
        self.arquitetura, self.taxa = arquitetura, taxa
        rng = np.random.default_rng(seed)
        self.pesos, self.vieses = [], []
        for entrada, saida in zip(arquitetura[:-1], arquitetura[1:]):
            limite = np.sqrt(6 / (entrada + saida))  # Xavier
            self.pesos.append(rng.uniform(-limite, limite, (entrada, saida)))
            self.vieses.append(np.zeros((1, saida)))

    @staticmethod
    def sigmoide(z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def forward(self, X):
        ativacoes = [X]
        for i, (W, b) in enumerate(zip(self.pesos, self.vieses)):
            z = ativacoes[-1] @ W + b
            a = self.sigmoide(z) if i == len(self.pesos)-1 else np.tanh(z)
            ativacoes.append(a)
        return ativacoes[-1], ativacoes

    @staticmethod
    def perda(y, y_pred):
        y = y.reshape(-1, 1)
        p = np.clip(y_pred, 1e-12, 1-1e-12)
        return float(-np.mean(y*np.log(p) + (1-y)*np.log(1-p)))

    def backward(self, y, ativacoes):
        y = y.reshape(-1, 1)
        n = len(y)
        dW = [np.empty_like(W) for W in self.pesos]
        db = [np.empty_like(b) for b in self.vieses]
        delta = ativacoes[-1] - y
        for camada in range(len(self.pesos)-1, -1, -1):
            dW[camada] = ativacoes[camada].T @ delta / n
            db[camada] = delta.mean(axis=0, keepdims=True)
            if camada > 0:
                delta = (delta @ self.pesos[camada].T) * (1-ativacoes[camada]**2)
        return dW, db

    def treinar(self, X, y, epocas=3000, intervalo=100):
        historico = []
        for epoca in range(epocas + 1):
            y_pred, ativacoes = self.forward(X)
            if epoca % intervalo == 0:
                historico.append((epoca, self.perda(y, y_pred)))
            if epoca == epocas:
                break
            dW, db = self.backward(y, ativacoes)
            for i in range(len(self.pesos)):
                self.pesos[i] -= self.taxa * dW[i]
                self.vieses[i] -= self.taxa * db[i]
        return np.array(historico)

    def prever_probabilidade(self, X):
        return self.forward(X)[0].ravel()

    def prever(self, X):
        return (self.prever_probabilidade(X) >= 0.5).astype(int)

## 3. Treinamento e avaliação

O teste fica separado durante todo o treinamento e é usado apenas na avaliação final.

In [ ]:
rede = RedeNeuralManual(arquitetura=(2, 8, 4, 1), taxa=0.05, seed=SEED)
historico = rede.treinar(X_treino, y_treino, epocas=3000, intervalo=50)

acc_treino = np.mean(rede.prever(X_treino) == y_treino)
acc_teste = np.mean(rede.prever(X_teste) == y_teste)
print(f'Acurácia de treino: {acc_treino:.2%}')
print(f'Acurácia de teste:  {acc_teste:.2%}')

plt.figure(figsize=(7, 4))
plt.plot(historico[:, 0], historico[:, 1])
plt.title('Curva de aprendizagem'); plt.xlabel('Época'); plt.ylabel('Entropia cruzada')
plt.grid(alpha=.25); plt.show()

In [ ]:
# Fronteira de decisão nos dados padronizados
X_total = np.vstack([X_treino, X_teste])
y_total = np.concatenate([y_treino, y_teste])
gx, gy = np.meshgrid(
    np.linspace(X_total[:,0].min()-.6, X_total[:,0].max()+.6, 350),
    np.linspace(X_total[:,1].min()-.6, X_total[:,1].max()+.6, 350)
)
grade = np.c_[gx.ravel(), gy.ravel()]
p = rede.prever_probabilidade(grade).reshape(gx.shape)
plt.figure(figsize=(7, 5))
plt.contourf(gx, gy, p, levels=np.linspace(0, 1, 21), cmap='RdBu_r', alpha=.65)
plt.contour(gx, gy, p, levels=[.5], colors='black')
plt.scatter(X_total[:,0], X_total[:,1], c=y_total, cmap='bwr', edgecolors='white', s=25)
plt.colorbar(label='P(classe = 1)'); plt.title('Fronteira de decisão aprendida')
plt.xlabel('Atributo 1'); plt.ylabel('Atributo 2'); plt.show()

## Conclusão

A rede aprende uma fronteira não linear usando apenas operações matriciais e a regra da cadeia. Em relação ao exemplo original, o experimento acrescenta uma camada oculta, troca a ativação interna, usa uma perda própria para classificação e mede generalização em um conjunto de teste. Como continuação, a mesma base matemática pode ser adaptada para regressão de glicemia futura, substituindo a saída sigmoide por uma saída linear e respeitando a separação temporal dos dados.